
# Social Contagion in Networks: Simple vs. Complex Contagion

### Learning goals
- Precisely define *simple* vs *complex* contagion in social systems.
- Implement canonical models: **Independent Cascade (IC)** for simple contagion and **Watts Threshold Model (WTM)** for complex contagion.
- Diagnose how network structure (degree heterogeneity, clustering, community structure, small-world rewiring) shapes cascade size and speed.
- Explore seeding strategies and parameter regimes that enable (or suppress) global cascades.
- Analyze the impact of individual thresholds and population heterogeneity on contagion dynamics.
- Investigate contagion in temporal, multiplex, directed, and weighted networks.
- Model competing contagions and interference effects.
- Compare influence maximization and seeding strategies.
- Connect simulations to analytical heuristics (percolation mapping for IC; vulnerable cluster condition for WTM).
- Extend models to spatial, agent-based, and hybrid frameworks.
- Visualize contagion processes with advanced techniques and animations.
- Apply models to empirical network data and discuss open problems and research directions.




## 0. Setup


In [ ]:

# Core scientific stack
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from collections import deque
from dataclasses import dataclass
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple, Union, Set

# Reproducibility
rng = np.random.default_rng(42)

# Matplotlib defaults (no explicit color choices per course tooling conventions)
plt.rcParams['figure.figsize'] = (6,4)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['axes.grid'] = True

# Helper to silence warnings in demos if needed
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)



## 1. Conceptual overview: simple vs. complex contagion

- **Simple contagion**: One *effective* exposure may be sufficient for adoption.  
  Canonical implementations include SI/SIR-style transmission or the **Independent Cascade (IC)** model (discrete-time: each new adopter gets *one* chance to activate each susceptible neighbor with probability \(p\)).

- **Complex contagion**: Adoption requires **reinforcement**—multiple concordant exposures (or a sufficiently large *fraction* of a node's neighbors) are needed before adoption is likely.  
  Canonical implementation: **Watts Threshold Model (WTM)** where node \(i\) adopts when
  \[ \frac{\#\{\text{adopted neighbors of } i\}}{k_i} \ge \phi_i, \]
  with \(\phi_i\in[0,1]\) a (possibly heterogeneous) threshold.
  
**Consequences for networks**
- Long ties and shortcuts **accelerate** simple contagion; they can **hinder** complex contagion by bypassing local reinforcement.
- Clustering/community structure can **delay** simple contagion but **enable** complex contagion via repeated exposures.
- Degree heterogeneity and seeding patterns strongly modulate cascade likelihood in both regimes.



## 2. Models and update rules

### 2.1 Independent Cascade (IC) — *simple contagion*
- Discrete synchronous time steps \(t=0,1,2,\dots\).
- Newly adopted nodes at time \(t\) get **one chance** to activate each susceptible neighbor with probability \(p\).
- Activated neighbors join at \(t+1\). No de-adoption.

### 2.2 Watts Threshold Model (WTM) — *complex contagion*
- Synchronous updates. Node \(i\) adopts when the adopted-neighbor fraction \(\ge \phi_i\).
- Thresholds \(\phi_i\) may be constant, drawn from a distribution, or depend on degree.
- Variants: integer \(k\)-thresholds, partial memory windows, weighted edges, multiplex reinforcement.



## 3. Implementation (networkx)
We'll implement clean, testable functions for IC (simple) and WTM (complex), plus utilities for seeding, sweeping parameters, and logging outcomes.


In [ ]:

@dataclass
class CascadeResult:
    adopted: Set
    history: List[Set]
    adopters_by_t: List[int]
    exposures: Dict # node -> number of received attempts (IC) or neighbor counts (WTM)
    steps: int

def choose_seeds(G: nx.Graph, k: int, method: str = "random", rng: Optional[np.random.Generator] = None) -> List:
    """Pick k seed nodes by a given strategy.
    
    method in {"random", "degree", "kcore"}.
    """
    rng = rng or np.random.default_rng()
    nodes = list(G.nodes())
    if k <= 0:
        return []
    if method == "random":
        return list(rng.choice(nodes, size=min(k, len(nodes)), replace=False))
    elif method == "degree":
        return [n for n,_ in sorted(G.degree(), key=lambda t: t[1], reverse=True)[:k]]
    elif method == "kcore":
        kc = nx.core_number(G)
        return [n for n,_ in sorted(kc.items(), key=lambda t: t[1], reverse=True)[:k]]
    else:
        raise ValueError(f"Unknown seeding method: {method}")


In [ ]:

def simulate_ic(
    G: nx.Graph,
    seeds: Iterable,
    p: float = 0.1,
    max_steps: Optional[int] = None,
    rng: Optional[np.random.Generator] = None,
) -> CascadeResult:
    """Independent Cascade (IC) model.
    
    Parameters
    ----------
    G : nx.Graph
    seeds : iterable of seed nodes
    p : float in [0,1], activation prob along each edge (one attempt per newly adopted neighbor)
    max_steps : optional step cap
    rng : numpy Generator
    
    Returns
    -------
    CascadeResult with adopted set, history (set per t), adopters_by_t (counts), exposures, steps
    """
    rng = rng or np.random.default_rng()
    adopted = set(seeds)
    frontier = set(seeds)  # newly adopted at current t
    history = [set(frontier)]
    adopters_by_t = [len(frontier)]
    exposures = {n: 0 for n in G.nodes()}  # number of *attempts* received
    
    steps = 0
    while frontier and (max_steps is None or steps < max_steps):
        steps += 1
        new_frontier = set()
        for u in frontier:
            for v in G.neighbors(u):
                if v in adopted:
                    continue
                exposures[v] = exposures.get(v, 0) + 1
                if rng.random() < p:
                    adopted.add(v)
                    new_frontier.add(v)
        if not new_frontier:
            break
        history.append(set(new_frontier))
        adopters_by_t.append(len(new_frontier))
        frontier = new_frontier
    return CascadeResult(adopted=adopted, history=history, adopters_by_t=adopters_by_t, exposures=exposures, steps=steps)


In [ ]:

def _resolve_thresholds(G: nx.Graph, phi: Union[float, Dict, Callable, Sequence], rng: Optional[np.random.Generator] = None) -> Dict:
    """Return dict of node->threshold in [0,1].
    - float: constant
    - dict: per-node
    - callable: called per node to sample threshold
    - sequence: aligned in order of G.nodes()
    """
    rng = rng or np.random.default_rng()
    nodes = list(G.nodes())
    if isinstance(phi, (float, int)):
        return {n: float(phi) for n in nodes}
    if isinstance(phi, dict):
        # default to 1.1 (impossible) if missing, ensuring explicitness
        return {n: float(phi.get(n, 1.1)) for n in nodes}
    if callable(phi):
        return {n: float(phi(n, G, rng)) for n in nodes}
    # sequence-like
    arr = list(map(float, phi))
    if len(arr) != len(nodes):
        raise ValueError("Length of phi sequence must equal number of nodes")
    return {n: arr[i] for i, n in enumerate(nodes)}

def simulate_wtm(
    G: nx.Graph,
    seeds: Iterable,
    phi: Union[float, Dict, Callable, Sequence] = 0.2,
    synchronous: bool = True,
    max_steps: Optional[int] = None,
    rng: Optional[np.random.Generator] = None,
) -> CascadeResult:
    """Watts Threshold Model (WTM) / fractional threshold complex contagion.
    
    Parameters
    ----------
    G : nx.Graph
    seeds : iterable of seed nodes
    phi : threshold spec (float, dict, callable, or sequence). Fraction of neighbors required.
    synchronous : if True, update in rounds; otherwise adopt as soon as threshold satisfied in a wave
    max_steps : optional step cap
    rng : numpy Generator
    """
    rng = rng or np.random.default_rng()
    thresholds = _resolve_thresholds(G, phi, rng=rng)
    adopted = set(seeds)
    history = [set(adopted)]
    adopters_by_t = [len(adopted)]
    steps = 0
    # track neighbor counts for introspection
    exposures = {n: 0 for n in G.nodes()}  # here we store current number of adopted neighbors
    
    def fraction_adopted_neighbors(v):
        k = G.degree(v)
        if k == 0:
            return 0.0
        a = sum((1 for u in G.neighbors(v) if u in adopted))
        exposures[v] = a  # store raw count
        return a / k
    
    while True and (max_steps is None or steps < max_steps):
        steps += 1
        if synchronous:
            new_adopters = {v for v in G.nodes() if v not in adopted and fraction_adopted_neighbors(v) >= thresholds[v]}
        else:
            # asynchronous "wave": scan in random order
            new_adopters = set()
            for v in rng.permutation(list(G.nodes())):
                if v in adopted or v in new_adopters:
                    continue
                if fraction_adopted_neighbors(v) >= thresholds[v]:
                    new_adopters.add(v)
                    adopted.add(v)  # immediate reinforcement effect
        if not new_adopters:
            break
        adopted |= new_adopters
        history.append(set(new_adopters))
        adopters_by_t.append(len(new_adopters))
    return CascadeResult(adopted=adopted, history=history, adopters_by_t=adopters_by_t, exposures=exposures, steps=steps)


In [ ]:

def make_graph(kind: str = "ER", n: int = 200, avg_k: float = 6, rng: Optional[np.random.Generator] = None) -> nx.Graph:
    """Convenience graph factory for demos: ER, WS, BA.
    avg_k is interpreted per model.
    """
    rng = rng or np.random.default_rng()
    if kind.upper() == "ER":
        p = avg_k / (n-1)
        return nx.erdos_renyi_graph(n, p, seed=rng)
    if kind.upper() == "WS":
        k = int(round(avg_k))
        k = k + (k % 2)  # ensure even
        # beta (rewiring prob) exposed in wrapper below
        return nx.watts_strogatz_graph(n, k, 0.1, seed=rng)  # default beta=0.1
    if kind.upper() == "BA":
        m = max(1, int(round(avg_k/2)))
        return nx.barabasi_albert_graph(n, m, seed=rng)
    raise ValueError("kind must be one of {'ER','WS','BA'}")

def run_many_ic(G, seeds, ps: Sequence[float], runs_per_p: int = 20, rng: Optional[np.random.Generator] = None) -> pd.DataFrame:
    rng = rng or np.random.default_rng()
    rows = []
    for p in ps:
        for r in range(runs_per_p):
            res = simulate_ic(G, seeds=seeds, p=p, rng=rng)
            rows.append(dict(model="IC", p=p, run=r, final_frac=len(res.adopted)/G.number_of_nodes(), steps=res.steps))
    return pd.DataFrame(rows)

def run_many_wtm(G, seeds, phis: Sequence[float], runs_per_phi: int = 1, rng: Optional[np.random.Generator] = None) -> pd.DataFrame:
    rng = rng or np.random.default_rng()
    rows = []
    for phi in phis:
        for r in range(runs_per_phi):
            res = simulate_wtm(G, seeds=seeds, phi=phi, rng=rng)
            rows.append(dict(model="WTM", phi=phi, run=r, final_frac=len(res.adopted)/G.number_of_nodes(), steps=res.steps))
    return pd.DataFrame(rows)

def plot_sweep(df: pd.DataFrame, x: str, y: str = "final_frac", xlabel: str = "", ylabel: str = "Final adoption"):
    ax = df.groupby(x)[y].mean().plot(marker="o", lw=1)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0,1)
    plt.show()



## 4. Quick sanity check
We run tiny demos to ensure the functions behave sensibly (no heavy computation here).


In [ ]:

# Tiny graphs to keep this cell fast
G_er = make_graph("ER", n=150, avg_k=6, rng=rng)
seed = choose_seeds(G_er, 3, method="degree", rng=rng)

# IC sweep
ps = np.linspace(0.01, 0.3, 10)
df_ic = run_many_ic(G_er, seeds=seed, ps=ps, runs_per_p=5, rng=rng)
plot_sweep(df_ic, x="p", xlabel="p (IC edge activation prob)")

# WTM sweep
phis = np.linspace(0.05, 0.6, 12)
df_wtm = run_many_wtm(G_er, seeds=seed, phis=phis, runs_per_phi=1, rng=rng)
plot_sweep(df_wtm, x="phi", xlabel="φ (WTM threshold)")



## 5. How structure shapes cascades
We will revisit three axes:
1. **Degree heterogeneity** (ER vs BA).
2. **Clustering / small-world** (WS with varying rewiring \( \beta \)).
3. **Community structure** (e.g., planted partition; optional extension).

**Your turn (guided):**
- For fixed \(n, \langle k \rangle\), compare IC vs WTM final adoption on ER vs BA across parameter sweeps.
- For WS, vary \( \beta \in [0,1] \) and show: IC global adoption typically rises with \( \beta \); WTM often shows a peak at *intermediate* \( \beta \) due to a balance of reinforcement and reach.


## 6. Thresholds, Heterogeneity, and Adoption Dynamics

**Outline:**
- Individual adoption thresholds
- Heterogeneous populations
- Impact on contagion dynamics

**Introduction:**
In real social systems, individuals differ in their willingness to adopt new ideas or behaviors. This section explores how varying thresholds and heterogeneity affect the spread of contagion, revealing richer and more realistic dynamics than uniform models.

**Motivation:**
Understanding threshold effects helps explain why some ideas fail to spread while others go viral, and how diversity in populations can stabilize or destabilize contagion.


In [ ]:
# Example: Simulating threshold adoption
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

G = nx.erdos_renyi_graph(20, 0.2)
thresholds = np.random.uniform(0.2, 0.8, size=G.number_of_nodes())
adopted = np.zeros(G.number_of_nodes(), dtype=bool)
adopted[0] = True  # seed

for step in range(5):
    for node in G.nodes():
        if not adopted[node]:
            neighbors = list(G.neighbors(node))
            if neighbors:
                frac_adopted = np.mean(adopted[neighbors])
                if frac_adopted >= thresholds[node]:
                    adopted[node] = True
    nx.draw(G, node_color=['red' if adopted[n] else 'blue' for n in G.nodes()], with_labels=True)
    plt.title(f'Step {step+1}')
    plt.show()

## 7. Temporal Networks and Time-Varying Contagion

**Outline:**
- Networks with changing edges/nodes
- Contagion in dynamic environments
- Comparison to static networks

**Introduction:**
Many real-world networks evolve over time, with connections appearing and disappearing. This section introduces temporal networks and demonstrates how time-varying structure impacts contagion processes.

**Motivation:**
Studying temporal networks helps us understand phenomena like information bursts, viral trends, and the timing of interventions in dynamic systems.


In [ ]:
# Example: Temporal network contagion
import networkx as nx
import matplotlib.pyplot as plt

# Create a base network
G = nx.cycle_graph(10)

# Simulate time-varying edges
snapshots = []
for t in range(5):
    G_t = G.copy()
    if t % 2 == 0:
        G_t.add_edge(0, t)
    else:
        if G_t.has_edge(0, t-1):
            G_t.remove_edge(0, t-1)
    snapshots.append(G_t)
    nx.draw(G_t, with_labels=True)
    plt.title(f'Time {t}')
    plt.show()

## 8. Multiplex and Multilayer Networks

**Outline:**
- Multiple types of relationships (layers)
- Contagion across layers
- Layer interactions and synergy

**Introduction:**
Social contagion often spreads through different channels—friendship, work, online platforms. Multiplex networks model these layers, revealing how cross-layer interactions can accelerate or hinder contagion.

**Motivation:**
Understanding multiplexity helps design interventions and predict outcomes in systems with overlapping social structures.


In [ ]:
# Example: Multiplex network contagion
import networkx as nx
import matplotlib.pyplot as plt

# Layer 1: friendship
G1 = nx.erdos_renyi_graph(10, 0.3)
# Layer 2: work
G2 = nx.erdos_renyi_graph(10, 0.2)

# Visualize both layers
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
nx.draw(G1, ax=axes[0], with_labels=True, node_color='skyblue')
axes[0].set_title('Friendship Layer')
nx.draw(G2, ax=axes[1], with_labels=True, node_color='lightgreen')
axes[1].set_title('Work Layer')
plt.show()

## 9. Competing Contagions and Interference

**Outline:**
- Multiple ideas/memes/behaviors
- Competition, suppression, coexistence
- Visualization of outcomes

**Introduction:**
In many settings, several ideas or behaviors compete for adoption. This section models competing contagions, showing how interference and suppression can shape the final outcome.

**Motivation:**
Understanding competition is crucial for marketing, public health, and information warfare, where multiple influences vie for attention.


In [ ]:
# Example: Competing contagions
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

G = nx.erdos_renyi_graph(15, 0.2)
state = np.zeros(G.number_of_nodes(), dtype=int)  # 0: susceptible, 1: A, 2: B
state[0] = 1  # seed A
state[-1] = 2  # seed B

for step in range(6):
    for node in G.nodes():
        if state[node] == 0:
            neighbors = list(G.neighbors(node))
            neighbor_states = [state[n] for n in neighbors]
            if neighbor_states.count(1) > neighbor_states.count(2):
                state[node] = 1
            elif neighbor_states.count(2) > neighbor_states.count(1):
                state[node] = 2
    colors = ['gray' if s==0 else 'red' if s==1 else 'blue' for s in state]
    nx.draw(G, node_color=colors, with_labels=True)
    plt.title(f'Step {step+1}')
    plt.show()

## 10. Influence Maximization and Seeding Strategies

**Outline:**
- Selecting optimal seed nodes
- Greedy and heuristic algorithms
- Comparison of outcomes

**Introduction:**
Influence maximization seeks to identify the best nodes to initiate contagion for maximum spread. This section introduces key algorithms and compares their effectiveness in different networks.

**Motivation:**
Optimizing seed selection is vital for viral marketing, information campaigns, and epidemic control.


In [ ]:
# Example: Influence maximization (degree heuristic)
import networkx as nx
import matplotlib.pyplot as plt

G = nx.erdos_renyi_graph(20, 0.15)
degrees = dict(G.degree())
seed = max(degrees, key=degrees.get)

adopted = [False]*G.number_of_nodes()
adopted[seed] = True

for step in range(5):
    for node in G.nodes():
        if not adopted[node]:
            neighbors = list(G.neighbors(node))
            if any(adopted[n] for n in neighbors):
                adopted[node] = True
    nx.draw(G, node_color=['red' if adopted[n] else 'blue' for n in G.nodes()], with_labels=True)
    plt.title(f'Step {step+1}')
    plt.show()

## 11. Empirical Data Analysis

**Outline:**
- Load real-world network data
- Analyze and visualize contagion events
- Compare empirical and simulated results

**Introduction:**
Theory meets reality in this section, where we use actual social network data to study contagion. By comparing empirical and simulated outcomes, we gain insight into the strengths and limitations of our models.

**Motivation:**
Empirical analysis validates models and uncovers new patterns in real social systems.


In [ ]:
# Example: Load and visualize empirical network data
import networkx as nx
import matplotlib.pyplot as plt

# Load edge list from file (replace with actual path)
# G = nx.read_edgelist('data/empirical_network.edgelist')
G = nx.karate_club_graph()  # Example built-in dataset
nx.draw(G, with_labels=True, node_color='orange')
plt.title('Empirical Network Example')
plt.show()

## 12. Contagion on Directed/Weighted Networks

**Outline:**
- Directionality and edge weights
- Impact on contagion
- Visualization and analysis

**Introduction:**
Not all connections are equal—some are stronger, and some have direction. This section explores how directed and weighted edges change the dynamics of contagion.

**Motivation:**
Accounting for direction and weight is essential for modeling influence, trust, and information flow in real networks.


In [ ]:
# Example: Directed and weighted contagion
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
G.add_edge(0, 1, weight=2)
G.add_edge(1, 2, weight=1)
G.add_edge(2, 0, weight=3)

pos = nx.spring_layout(G)
weights = [G[u][v]['weight'] for u,v in G.edges()]
nx.draw(G, pos, with_labels=True, node_color='yellow', width=weights, arrows=True)
plt.title('Directed & Weighted Network')
plt.show()

## 13. Control, Intervention, and Immunization

**Outline:**
- Strategies to slow, stop, or redirect contagion
- Node/edge removal, targeted immunization
- Visualization of intervention effects

**Introduction:**
Controlling contagion is crucial in public health, cybersecurity, and rumor management. This section explores intervention strategies and their impact on the spread.

**Motivation:**
Effective interventions can prevent epidemics, limit misinformation, and protect vulnerable populations.


In [ ]:
# Example: Intervention by node removal
import networkx as nx
import matplotlib.pyplot as plt

G = nx.erdos_renyi_graph(15, 0.2)
critical = max(dict(G.degree()).items(), key=lambda x: x[1])[0]
G_removed = G.copy()
G_removed.remove_node(critical)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
nx.draw(G, ax=axes[0], with_labels=True, node_color='red')
axes[0].set_title('Original Network')
nx.draw(G_removed, ax=axes[1], with_labels=True, node_color='blue')
axes[1].set_title('After Node Removal')
plt.show()

## 14. Contagion Beyond Networks: Spatial, Agent-Based, and Hybrid Models

**Outline:**
- Spatial grids and agent-based models
- Hybrid approaches
- Comparison with network-based contagion

**Introduction:**
Contagion can also be modeled in spatial and agent-based frameworks, capturing local interactions and movement. This section introduces these models and compares them to network-based approaches.

**Motivation:**
Spatial and agent-based models provide deeper insight into phenomena like disease spread, crowd behavior, and innovation diffusion.


In [ ]:
# Example: Simple agent-based contagion on a grid
import numpy as np
import matplotlib.pyplot as plt

size = 10
grid = np.zeros((size, size), dtype=int)
grid[size//2, size//2] = 1  # seed

for step in range(5):
    new_grid = grid.copy()
    for i in range(size):
        for j in range(size):
            if grid[i, j] == 0:
                neighbors = [grid[x, y] for x in [i-1, i, i+1] for y in [j-1, j, j+1]
                             if 0 <= x < size and 0 <= y < size and (x != i or y != j)]
                if any(n == 1 for n in neighbors):
                    new_grid[i, j] = 1
    grid = new_grid
    plt.imshow(grid, cmap='Reds')
    plt.title(f'Step {step+1}')
    plt.show()

## 15. Advanced Visualization and Animation

**Outline:**
- Interactive widgets and animations
- Real-time and step-by-step visualizations
- GIF creation

**Introduction:**
Visualizing contagion dynamically brings models to life. This section introduces advanced visualization tools and techniques for interactive and animated exploration.

**Motivation:**
Animations and interactivity enhance understanding and engagement, making complex processes intuitive.


In [ ]:
# Example: Animated contagion (matplotlib animation)
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.animation as animation

G = nx.erdos_renyi_graph(15, 0.2)
adopted = [False]*G.number_of_nodes()
adopted[0] = True

fig, ax = plt.subplots()
pos = nx.spring_layout(G)

frames = []
for step in range(6):
    for node in G.nodes():
        if not adopted[node]:
            neighbors = list(G.neighbors(node))
            if any(adopted[n] for n in neighbors):
                adopted[node] = True
    frame = nx.draw(G, pos, node_color=['red' if adopted[n] else 'blue' for n in G.nodes()], with_labels=True, ax=ax)
    frames.append([frame])
ani = animation.ArtistAnimation(fig, frames, interval=800, blit=True)
plt.show()

## 16. Open Problems and Research Directions

**Outline:**
- Current challenges in social contagion modeling
- Open questions and future directions
- Opportunities for research and application

**Introduction:**
Social contagion remains a vibrant field with many unsolved problems. This section highlights open questions and suggests directions for future research and practical impact.

**Motivation:**
Identifying open problems inspires innovation and guides the next generation of research in social contagion.


## 17. Social Contagion in LLMs

**Outline:**
- LLMs as agents in social networks
- Contagion of ideas, behaviors, and memes in LLM societies
- Simulation and analysis of propagation dynamics

**Introduction:**
Large Language Models (LLMs) can act as agents in digital societies, propagating information, behaviors, and even biases. This section explores how social contagion manifests in networks of LLMs, including emergent phenomena and feedback loops.

**Motivation:**
Understanding contagion in LLM agent societies is crucial for designing robust, ethical, and interpretable AI systems, and for anticipating unintended consequences in large-scale deployments.


In [ ]:
# Example: Simulating idea spread in LLM agent society
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

# Each node is an LLM agent
G = nx.erdos_renyi_graph(12, 0.3)
ideas = np.zeros(G.number_of_nodes(), dtype=int)
ideas[0] = 1  # seed idea in agent 0

for step in range(6):
    for node in G.nodes():
        if ideas[node] == 0:
            neighbors = list(G.neighbors(node))
            if any(ideas[n] == 1 for n in neighbors):
                # LLM agent adopts idea with some probability
                if np.random.rand() < 0.5:
                    ideas[node] = 1
    nx.draw(G, node_color=['red' if ideas[n] else 'blue' for n in G.nodes()], with_labels=True)
    plt.title(f'LLM Society, Step {step+1}')
    plt.show()

## 18. Percolation & Generating-Function Derivations (Math Core)

**Outline:**
- Percolation theory and global cascades
- Generating functions for cascade size
- Analytical derivations and phase transitions

**Introduction:**
Percolation theory provides a mathematical foundation for understanding global cascades in networks. Generating functions allow us to derive expected cascade sizes and identify critical points for phase transitions.

**Motivation:**
Analytical tools deepen our understanding of when and how large-scale contagion occurs, complementing simulation-based approaches.


In [ ]:
# Example: Percolation threshold in ER graph
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

n = 200
ps = np.linspace(0, 0.05, 20)
final_fracs = []
for p in ps:
    G = nx.erdos_renyi_graph(n, p)
    largest_cc = max(nx.connected_components(G), key=len) if G.number_of_edges() > 0 else set()
    final_fracs.append(len(largest_cc)/n)
plt.plot(ps, final_fracs, marker='o')
plt.xlabel('Edge probability p')
plt.ylabel('Largest component fraction')
plt.title('Percolation Threshold in ER Graph')
plt.show()

## 19. Message Passing & Edge-Based Compartmental Models

**Outline:**
- Message passing for contagion dynamics
- Edge-based compartmental models
- Applications to epidemic and information spread

**Introduction:**
Message passing and edge-based models provide powerful frameworks for analyzing contagion, especially in large and heterogeneous networks. These approaches can efficiently compute expected outcomes and track the spread at the edge level.

**Motivation:**
These models are widely used in epidemiology and network science for their scalability and analytical tractability.


In [ ]:
# Example: Edge-based SIR model (sketch)
import networkx as nx
import numpy as np

G = nx.erdos_renyi_graph(20, 0.2)
status = {n: 'S' for n in G.nodes()}
status[0] = 'I'  # seed infection

for step in range(5):
    new_status = status.copy()
    for u in G.nodes():
        if status[u] == 'I':
            for v in G.neighbors(u):
                if status[v] == 'S' and np.random.rand() < 0.3:
                    new_status[v] = 'I'
            new_status[u] = 'R'  # recover
    status = new_status
    print(f'Step {step+1}:', dict(status))

## 20. Causal Identification: Contagion vs Homophily

**Outline:**
- Distinguishing contagion from homophily
- Experimental and statistical approaches
- Implications for inference and policy

**Introduction:**
Observed correlations in networks may arise from contagion (influence) or homophily (similarity). This section discusses methods to disentangle these effects and the challenges involved.

**Motivation:**
Causal identification is essential for valid inference and effective intervention in social systems.


In [ ]:

# TODO: WS study scaffold
def ws_sweep(n=400, k=6, betas=np.linspace(0.0, 1.0, 9), model="IC", param=0.1, seeds_k=3, rng=None):
    rng = rng or np.random.default_rng()
    rows = []
    for beta in betas:
        G = nx.watts_strogatz_graph(n, k + (k % 2), beta, seed=rng)
        seeds = choose_seeds(G, seeds_k, method="degree", rng=rng)
        if model.upper() == "IC":
            res = simulate_ic(G, seeds, p=float(param), rng=rng)
        else:
            res = simulate_wtm(G, seeds, phi=float(param), rng=rng)
        rows.append(dict(beta=beta, final_frac=len(res.adopted)/n, steps=res.steps))
    return pd.DataFrame(rows)

# Example (keep small by default)
df_ws_ic = ws_sweep(n=250, k=6, betas=np.linspace(0,1,8), model="IC", param=0.1, seeds_k=3, rng=rng)
plot_sweep(df_ws_ic, x="beta", xlabel="WS rewiring β")

df_ws_wtm = ws_sweep(n=250, k=6, betas=np.linspace(0,1,8), model="WTM", param=0.2, seeds_k=3, rng=rng)
plot_sweep(df_ws_wtm, x="beta", xlabel="WS rewiring β")



## 6. Seeding strategies
Compare **random**, **high-degree**, and **k-core** seeding under both IC and WTM.  
Key takeaways to look for:
- Under IC, high-degree/k-core often give larger cascades (more outward edges).
- Under WTM, seeds that are **clustered** can be more effective (local reinforcement).


In [ ]:

def compare_seeding(G, k: int = 3, model: str = "IC", param: float = 0.1, rng=None) -> pd.DataFrame:
    rng = rng or np.random.default_rng()
    rows = []
    for method in ["random", "degree", "kcore"]:
        seeds = choose_seeds(G, k, method=method, rng=rng)
        if model.upper() == "IC":
            res = simulate_ic(G, seeds, p=param, rng=rng)
        else:
            res = simulate_wtm(G, seeds, phi=param, rng=rng)
        rows.append(dict(method=method, final_frac=len(res.adopted)/G.number_of_nodes(), steps=res.steps))
    return pd.DataFrame(rows)

# Demo on a single graph
G_demo = make_graph("WS", n=250, avg_k=6, rng=rng)
df_seed_ic = compare_seeding(G_demo, k=5, model="IC", param=0.12, rng=rng)
df_seed_wtm = compare_seeding(G_demo, k=5, model="WTM", param=0.25, rng=rng)

# Plot
ax = df_seed_ic.set_index("method")["final_frac"].plot(kind="bar", rot=0)
ax.set_title("IC: Final adoption by seeding method")
ax.set_ylim(0,1)
plt.show()

ax = df_seed_wtm.set_index("method")["final_frac"].plot(kind="bar", rot=0)
ax.set_title("WTM: Final adoption by seeding method")
ax.set_ylim(0,1)
plt.show()



## 7. Analytical touchpoints (brief)
- **IC ↔ bond percolation**: The final adoption set from a single seed at large \(t\) maps to the connected component in a graph where each edge is kept with prob \(p\). The *global cascade* condition on sparse random graphs parallels the giant component condition \( \langle k \rangle > 1 \) and, more generally, \( \kappa = \frac{\langle k^2\rangle}{\langle k\rangle} > 2 \) for configuration models.
- **WTM vulnerable cluster**: Nodes with \( \phi_i < 1/k_i \) are *vulnerable* (adopt if any neighbor adopts). A global cascade requires a supercritical branching of vulnerable nodes seeded by the initial shock; clustering can grow vulnerable components via reinforcement beyond tree-like assumptions.



## 8. Extensions & exercises
- **Temporal reinforcement windows**: require \(m\) exposures within \(W\) time steps.
- **Weighted or directed graphs**: use weighted fractions or in-neighbors only.
- **Multiplex**: require thresholds per layer or cross-layer reinforcement (e.g., adopt if fraction in *any* of two layers clears \(\phi\), or if the *sum* across layers does).
- **Your turn**: Implement an exposure-memory variant of WTM: a node adopts when cumulative exposures reach \(m\) *and* at least a fraction \(\phi\) of neighbors are adopted.



## 9. References (classic)
- Watts, D.J. (2002). *A simple model of global cascades on random networks*. PNAS.
- Centola, D., & Macy, M. (2007). *Complex contagions and the weakness of long ties*. AJS.
- Centola, D. (2010). *The Spread of Behavior in an Online Social Network Experiment*. Science.
- Kempe, D., Kleinberg, J., & Tardos, É. (2003). *Maximizing the spread of influence through a social network*. KDD.
- Gleeson, J.P. & Cahalane, D.J. (2007); Gleeson (2008). *Cascades on random networks*. 
- Pastor-Satorras, R., Castellano, C., Van Mieghem, P., & Vespignani, A. (2015). *Epidemic processes in complex networks*. Rev. Mod. Phys.
